# Energy-Aware Job-Shop Scheduling (NP-Hard)
## ShinkaEvolve Integration

This notebook defines the problem instance, evaluation harness, and evolutionary operators
for a government data center scheduling optimization problem.

In [5]:
"""
Problem Instance Definition
===========================
"""
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Any, Callable
import random
import copy

# =============================================================================
# CONSTANTS
# =============================================================================

TIME_HORIZON = 24  # hours
TIME_UNIT = 1      # 1 hour per slot

ENERGY_PRICE = {  # Electricity price (¥ / kWh) by hour
    0: 8, 1: 8, 2: 8, 3: 8, 4: 8, 5: 8,          # off-peak
    6: 18, 7: 18, 8: 18, 9: 18,                  # morning peak
    10: 12, 11: 12, 12: 12, 13: 12, 14: 12, 15: 12, 16: 12,
    17: 22, 18: 22, 19: 22, 20: 22, 21: 22,      # evening peak
    22: 10, 23: 10                               # late night
}

# =============================================================================
# DATA STRUCTURES
# =============================================================================

@dataclass
class Server:
    id: str
    server_type: str
    power_kw: float

@dataclass
class Operation:
    server_type: str
    duration: int

@dataclass
class Job:
    id: str
    description: str
    priority: str
    deadline: int
    delay_weight: float
    operations: List[Operation]

@dataclass
class Instance:
    """Complete problem instance."""
    servers: Dict[str, Server]
    jobs: Dict[str, Job]
    energy_price: Dict[int, float]
    time_horizon: int
    weights: Dict[str, float]

# Schedule type: job_id -> [(server_id, start_time), ...]
Schedule = Dict[str, List[Dict[str, Any]]]

# =============================================================================
# CREATE INSTANCE
# =============================================================================

def create_instance() -> Instance:
    """Create the problem instance."""
    servers = {
        "S1": Server("S1", "CPU", 2.0),
        "S2": Server("S2", "CPU", 2.0),
        "S3": Server("S3", "GPU", 4.5),
        "S4": Server("S4", "Storage", 1.2),
    }
    
    jobs = {
        "J1": Job("J1", "Emergency response analytics", "HIGH", 8, 10.0, [
            Operation("Storage", 1), Operation("CPU", 2), Operation("GPU", 1)
        ]),
        "J2": Job("J2", "Disaster damage estimation", "HIGH", 10, 8.0, [
            Operation("Storage", 1), Operation("GPU", 2), Operation("CPU", 1)
        ]),
        "J3": Job("J3", "Tax data aggregation", "MEDIUM", 18, 4.0, [
            Operation("CPU", 2), Operation("Storage", 1), Operation("CPU", 1)
        ]),
        "J4": Job("J4", "Census preprocessing", "MEDIUM", 20, 3.0, [
            Operation("Storage", 1), Operation("CPU", 2)
        ]),
        "J5": Job("J5", "Open data batch update", "LOW", 22, 1.0, [
            Operation("CPU", 1), Operation("Storage", 1)
        ]),
        "J6": Job("J6", "Archive and backup", "LOW", 24, 0.5, [
            Operation("Storage", 2), Operation("CPU", 2)
        ]),
    }
    
    weights = {"energy_cost": 1.0, "delay_penalty": 5.0}
    
    return Instance(servers, jobs, ENERGY_PRICE, TIME_HORIZON, weights)

# Create the instance
INSTANCE = create_instance()
print(f"Created instance with {len(INSTANCE.jobs)} jobs and {len(INSTANCE.servers)} servers")

Created instance with 6 jobs and 4 servers


# Fitness Evaluation

In [7]:
"""
Fitness Evaluation
==================
Evaluates a schedule and returns fitness score.
Lower is better (0 = perfect).
"""

@dataclass
class Fitness:
    """Evaluation result."""
    score: float
    energy_cost: float
    delay_penalty: float
    violations: List[str]
    job_completion_times: Dict[str, int]
    
    def is_valid(self) -> bool:
        return len(self.violations) == 0


def evaluate(inst: Instance, schedule: Schedule) -> Fitness:
    """
    Evaluate a schedule for the job-shop problem.
    Returns Fitness object (lower score = better).
    """
    server_usage = {s: [] for s in inst.servers}
    job_completion_times = {}
    total_energy_cost = 0.0
    total_delay_penalty = 0.0
    violations = []

    for job_id, ops in schedule.items():
        job = inst.jobs[job_id]
        
        if len(ops) != len(job.operations):
            violations.append(f"{job_id}: wrong number of operations")
            continue

        prev_end = None
        for idx, op in enumerate(ops):
            server_id = op["server"]
            start = op["start"]
            duration = job.operations[idx].duration
            end = start + duration

            # Job order constraint
            if prev_end is not None and start < prev_end:
                violations.append(f"{job_id}: op {idx} starts before previous ends")
            prev_end = end

            # Server type constraint
            required_type = job.operations[idx].server_type
            actual_type = inst.servers[server_id].server_type
            if required_type != actual_type:
                violations.append(f"{job_id}: op {idx} requires {required_type}, got {actual_type}")

            # Record server usage
            server_usage[server_id].append((start, end, job_id, idx))

            # Energy cost
            power = inst.servers[server_id].power_kw
            for t in range(start, end):
                total_energy_cost += power * inst.energy_price[t % inst.time_horizon]

        # Completion time & delay
        completion = prev_end if prev_end else 0
        job_completion_times[job_id] = completion
        delay = max(0, completion - job.deadline)
        total_delay_penalty += delay * job.delay_weight

    # Server capacity constraints (no overlap)
    for server_id, intervals in server_usage.items():
        intervals.sort()
        for i in range(len(intervals) - 1):
            _, end1, job1, _ = intervals[i]
            start2, _, job2, _ = intervals[i + 1]
            if start2 < end1:
                violations.append(f"{server_id}: overlap between {job1} and {job2}")

    # Compute objective
    score = (
        inst.weights["energy_cost"] * total_energy_cost
        + inst.weights["delay_penalty"] * total_delay_penalty
    )
    
    # Add heavy penalty for violations
    score += len(violations) * 10000

    return Fitness(
        score=round(score, 2),
        energy_cost=round(total_energy_cost, 2),
        delay_penalty=round(total_delay_penalty, 2),
        violations=violations,
        job_completion_times=job_completion_times
    )

print("Fitness evaluation function defined ✓")

Fitness evaluation function defined ✓


# Solver Interface & Baselines

In [9]:
"""
Solver Interface & Baseline Algorithms
======================================
"""

# Type alias: Solver takes Instance, returns Schedule
Solver = Callable[[Instance], Schedule]


def get_servers_by_type(inst: Instance, server_type: str) -> List[str]:
    """Get all server IDs of a given type."""
    return [s.id for s in inst.servers.values() if s.server_type == server_type]


def fifo_solver(inst: Instance) -> Schedule:
    """FIFO baseline: schedule jobs in order, earliest possible time."""
    server_available = {s: 0 for s in inst.servers}
    job_available = {j: 0 for j in inst.jobs}
    schedule = {}

    for job_id in sorted(inst.jobs.keys()):
        job = inst.jobs[job_id]
        schedule[job_id] = []

        for op in job.operations:
            compatible = get_servers_by_type(inst, op.server_type)
            chosen = min(compatible, key=lambda s: server_available[s])
            start = max(job_available[job_id], server_available[chosen])
            
            schedule[job_id].append({"server": chosen, "start": start})
            
            finish = start + op.duration
            job_available[job_id] = finish
            server_available[chosen] = finish

    return schedule


def energy_greedy_solver(inst: Instance) -> Schedule:
    """Energy-greedy: prefer low-cost time slots."""
    server_available = {s: 0 for s in inst.servers}
    job_available = {j: 0 for j in inst.jobs}
    schedule = {}

    def op_energy_cost(server_id: str, start: int, duration: int) -> float:
        power = inst.servers[server_id].power_kw
        return sum(power * inst.energy_price[t % inst.time_horizon] 
                   for t in range(start, start + duration))

    for job_id in sorted(inst.jobs.keys()):
        job = inst.jobs[job_id]
        schedule[job_id] = []

        for op in job.operations:
            compatible = get_servers_by_type(inst, op.server_type)
            best_choice, best_cost = None, float("inf")

            for server_id in compatible:
                earliest = max(job_available[job_id], server_available[server_id])
                # Try a range of start times (at least try earliest)
                max_start = max(earliest + 1, inst.time_horizon - op.duration + 1)
                for start in range(earliest, max_start):
                    cost = op_energy_cost(server_id, start, op.duration)
                    if cost < best_cost:
                        best_cost = cost
                        best_choice = (server_id, start)
            
            # Fallback if no choice found
            if best_choice is None:
                server_id = compatible[0]
                start = max(job_available[job_id], server_available[server_id])
            else:
                server_id, start = best_choice
            
            schedule[job_id].append({"server": server_id, "start": start})
            
            finish = start + op.duration
            job_available[job_id] = finish
            server_available[server_id] = finish

    return schedule


def priority_solver(inst: Instance) -> Schedule:
    """Priority-first: schedule high-priority jobs first."""
    priority_order = {"HIGH": 0, "MEDIUM": 1, "LOW": 2}
    sorted_jobs = sorted(inst.jobs.keys(), 
                        key=lambda j: (priority_order[inst.jobs[j].priority], j))
    
    server_available = {s: 0 for s in inst.servers}
    job_available = {j: 0 for j in inst.jobs}
    schedule = {}

    for job_id in sorted_jobs:
        job = inst.jobs[job_id]
        schedule[job_id] = []

        for op in job.operations:
            compatible = get_servers_by_type(inst, op.server_type)
            chosen = min(compatible, key=lambda s: server_available[s])
            start = max(job_available[job_id], server_available[chosen])
            
            schedule[job_id].append({"server": chosen, "start": start})
            
            finish = start + op.duration
            job_available[job_id] = finish
            server_available[chosen] = finish

    return schedule


# Test baselines
print("Testing baseline solvers...")
for name, solver in [("FIFO", fifo_solver), ("Energy-Greedy", energy_greedy_solver), ("Priority", priority_solver)]:
    sched = solver(INSTANCE)
    fit = evaluate(INSTANCE, sched)
    print(f"  {name}: score={fit.score}, energy={fit.energy_cost}, delay={fit.delay_penalty}, valid={fit.is_valid()}")

Testing baseline solvers...
  FIFO: score=508.0, energy=508.0, delay=0.0, valid=True
  Energy-Greedy: score=906.7, energy=359.2, delay=109.5, valid=True
  Priority: score=508.0, energy=508.0, delay=0.0, valid=True


# Evolutionary Operators

In [ ]:
"""
Evolutionary Operators
======================
Mutation and crossover for schedule evolution.
"""

def mutate_schedule(inst: Instance, schedule: Schedule, mutation_rate: float = 0.3) -> Schedule:
    """
    Mutate a schedule by randomly adjusting start times or swapping servers.
    """
    new_schedule = copy.deepcopy(schedule)
    
    for job_id in new_schedule:
        job = inst.jobs[job_id]
        for idx, op_sched in enumerate(new_schedule[job_id]):
            if random.random() < mutation_rate:
                op = job.operations[idx]
                compatible = get_servers_by_type(inst, op.server_type)
                
                # Mutate: change server or shift time
                if random.random() < 0.5 and len(compatible) > 1:
                    # Swap to different compatible server
                    op_sched["server"] = random.choice(compatible)
                else:
                    # Shift start time randomly
                    delta = random.randint(-3, 3)
                    op_sched["start"] = max(0, op_sched["start"] + delta)
    
    return new_schedule


def repair_schedule(inst: Instance, schedule: Schedule) -> Schedule:
    """
    Repair a schedule to satisfy precedence and capacity constraints.
    """
    new_schedule = {}
    server_available = {s: 0 for s in inst.servers}
    
    # Sort jobs by first operation's start time in original schedule
    job_order = sorted(schedule.keys(), 
                       key=lambda j: schedule[j][0]["start"] if schedule[j] else 0)
    
    for job_id in job_order:
        job = inst.jobs[job_id]
        new_schedule[job_id] = []
        job_time = 0
        
        for idx, op_sched in enumerate(schedule[job_id]):
            op = job.operations[idx]
            server_id = op_sched["server"]
            
            # Ensure server compatibility
            compatible = get_servers_by_type(inst, op.server_type)
            if server_id not in compatible:
                server_id = compatible[0]
            
            # Ensure valid start time (after prev op, after server available)
            requested_start = op_sched["start"]
            actual_start = max(requested_start, job_time, server_available[server_id])
            
            new_schedule[job_id].append({"server": server_id, "start": actual_start})
            
            finish = actual_start + op.duration
            job_time = finish
            server_available[server_id] = finish
    
    return new_schedule


def crossover(inst: Instance, parent1: Schedule, parent2: Schedule) -> Schedule:
    """
    Single-point crossover: take jobs from each parent.
    """
    child = {}
    jobs = list(inst.jobs.keys())
    crossover_point = random.randint(1, len(jobs) - 1)
    
    for i, job_id in enumerate(jobs):
        if i < crossover_point:
            child[job_id] = copy.deepcopy(parent1[job_id])
        else:
            child[job_id] = copy.deepcopy(parent2[job_id])
    
    return repair_schedule(inst, child)


print("Evolutionary operators defined ✓")

In [ ]:
"""
ShinkaEvolve - Single Iteration
===============================
Run one evolutionary iteration for algorithm development.
"""

@dataclass
class Individual:
    """An individual in the population."""
    schedule: Schedule
    fitness: Fitness


def run_one_iteration(
    inst: Instance,
    population: List[Individual],
    mutation_rate: float = 0.3,
    elite_count: int = 2
) -> Tuple[List[Individual], Dict[str, Any]]:
    """
    Run one iteration of evolutionary optimization.
    
    Steps:
    1. Evaluate all individuals
    2. Select elite (best performers)
    3. Generate offspring via crossover + mutation
    4. Replace population
    
    Returns:
        Updated population and iteration statistics
    """
    # Sort by fitness (lower is better)
    population.sort(key=lambda ind: ind.fitness.score)
    
    # Keep elite
    elite = population[:elite_count]
    
    # Generate offspring
    offspring = []
    while len(offspring) < len(population) - elite_count:
        # Tournament selection
        p1 = min(random.sample(population, 3), key=lambda x: x.fitness.score)
        p2 = min(random.sample(population, 3), key=lambda x: x.fitness.score)
        
        # Crossover
        child_sched = crossover(inst, p1.schedule, p2.schedule)
        
        # Mutation
        child_sched = mutate_schedule(inst, child_sched, mutation_rate)
        child_sched = repair_schedule(inst, child_sched)
        
        # Evaluate
        child_fit = evaluate(inst, child_sched)
        offspring.append(Individual(child_sched, child_fit))
    
    # New population = elite + offspring
    new_population = elite + offspring
    
    # Statistics
    scores = [ind.fitness.score for ind in new_population]
    stats = {
        "best_score": min(scores),
        "avg_score": sum(scores) / len(scores),
        "worst_score": max(scores),
        "best_valid": new_population[0].fitness.is_valid(),
    }
    
    return new_population, stats


def initialize_population(inst: Instance, size: int = 10) -> List[Individual]:
    """Create initial population from baselines + random mutations."""
    population = []
    
    # Seed with baselines
    for solver in [fifo_solver, energy_greedy_solver, priority_solver]:
        sched = solver(inst)
        fit = evaluate(inst, sched)
        population.append(Individual(sched, fit))
    
    # Fill rest with mutations of baselines
    while len(population) < size:
        base = random.choice(population[:3])
        mutated = mutate_schedule(inst, base.schedule, 0.5)
        repaired = repair_schedule(inst, mutated)
        fit = evaluate(inst, repaired)
        population.append(Individual(repaired, fit))
    
    return population


print("ShinkaEvolve iteration functions defined ✓")

# Run One Iteration

In [ ]:
"""
Run One Evolutionary Iteration
==============================
"""
random.seed(42)  # For reproducibility

# Initialize population
print("Initializing population...")
population = initialize_population(INSTANCE, size=10)

print(f"\nInitial Population ({len(population)} individuals):")
print("-" * 50)
for i, ind in enumerate(sorted(population, key=lambda x: x.fitness.score)):
    print(f"  {i+1}. Score: {ind.fitness.score:>8.2f} | Energy: {ind.fitness.energy_cost:>6.2f} | "
          f"Delay: {ind.fitness.delay_penalty:>6.2f} | Valid: {ind.fitness.is_valid()}")

# Run ONE iteration
print("\n" + "=" * 50)
print("Running ONE evolutionary iteration...")
print("=" * 50)

population, stats = run_one_iteration(INSTANCE, population, mutation_rate=0.3)

print(f"\nIteration Results:")
print(f"  Best Score:  {stats['best_score']:.2f}")
print(f"  Avg Score:   {stats['avg_score']:.2f}")
print(f"  Worst Score: {stats['worst_score']:.2f}")
print(f"  Best Valid:  {stats['best_valid']}")

# Show best solution
best = min(population, key=lambda x: x.fitness.score)
print(f"\nBest Schedule Found:")
print("-" * 50)
for job_id in sorted(best.schedule.keys()):
    ops = best.schedule[job_id]
    job = INSTANCE.jobs[job_id]
    op_str = " → ".join([f"{o['server']}@t={o['start']}" for o in ops])
    completion = best.fitness.job_completion_times.get(job_id, "?")
    print(f"  {job_id} ({job.priority}): {op_str} | Done: t={completion}, Deadline: {job.deadline}")

# Run Multiple Iterations (Optional)

In [ ]:
"""
Run Multiple Iterations
=======================
Extend to run multiple generations.
"""

def evolve(inst: Instance, generations: int = 10, population_size: int = 10) -> Individual:
    """Run full evolutionary optimization."""
    population = initialize_population(inst, population_size)
    
    print(f"Generation 0: best={min(p.fitness.score for p in population):.2f}")
    
    for gen in range(1, generations + 1):
        population, stats = run_one_iteration(inst, population)
        print(f"Generation {gen}: best={stats['best_score']:.2f}, avg={stats['avg_score']:.2f}")
    
    return min(population, key=lambda x: x.fitness.score)

# Example: run 5 generations
# best = evolve(INSTANCE, generations=5)
# print(f"\nFinal best score: {best.fitness.score}")

print("Multi-generation evolution function ready (uncomment to run)")